# Seminar 7: Recursion

**University of Geneva — Algorithmics & Data Structures**

---

> **Book reference:** Week 7 follows the **Recursion** chapter ([*Problem Solving with Algorithms and Data Structures using Python*](https://runestone.academy/ns/books/published/pythonds/Recursion/toctree.html)), numbered **Chapter 4** in the Runestone edition. Reading it alongside the seminar is strongly recommended.

## Learning Objectives

By the end of this seminar you will be able to:

- State the three laws of recursion and apply them to new problems
- Trace recursive calls using a call-stack diagram
- Implement factorial, base conversion, and Tower of Hanoi recursively
- Explain why naive Fibonacci is O(2ⁿ) and how memoisation reduces it to O(n)
- Know when recursion is the right tool — and when to prefer iteration

---

## Part 1: Theory


### 1.1 What is Recursion?

A function is **recursive** when it calls itself to solve a smaller version of the same problem. Each recursive call reduces the problem until a **base case** is reached — a version simple enough to solve directly without further recursion.

#### The Three Laws of Recursion

1. **A recursive algorithm must have a base case.** The base case stops the recursion. Without it, calls continue indefinitely and Python raises `RecursionError`.
2. **A recursive algorithm must change its state and move toward the base case.** Each call must make the problem strictly smaller (fewer elements, lower number, shorter string).
3. **A recursive algorithm must call itself (recursively).** This is what makes it recursive.

#### Analogy: Russian Nesting Dolls

Imagine a set of Russian nesting dolls (matryoshkas). Each doll contains a smaller, identical doll — until you reach the smallest one with nothing inside (the base case). Opening a doll is the recursive call; the smallest doll is the base case.

#### Iterative vs Recursive: List Sum

Both approaches produce the same result; recursion expresses the same logic in a self-referential way:

```python
# Iterative
def list_sum_iter(lst):
    total = 0
    for n in lst:
        total += n
    return total

# Recursive
def list_sum(lst):
    if len(lst) == 0:        # base case: empty list sums to 0
        return 0
    return lst[0] + list_sum(lst[1:])   # recursive step
```

The recursive version reads as: *the sum of a list is its first element plus the sum of the rest*.

In [ ]:
def list_sum(lst):
    """Return the sum of a list recursively."""
    if len(lst) == 0:
        return 0
    return lst[0] + list_sum(lst[1:])

# Verify against built-in
data = [1, 3, 5, 7, 9]
assert list_sum(data) == sum(data) == 25
assert list_sum([]) == 0
print("list_sum works correctly")

### 1.2 Simple Recursive Patterns

#### Countdown

```python
def countdown(n):
    if n <= 0:          # base case
        print("Go!")
        return
    print(n)
    countdown(n - 1)   # recursive step: n decreases each call
```

#### Factorial

n! = n × (n−1) × … × 1, with 0! = 1

```python
def factorial(n):
    if n == 0:          # base case
        return 1
    return n * factorial(n - 1)   # recursive step
```

#### String Reversal

```python
def reverse_str(s):
    if len(s) <= 1:    # base case: empty string or single char
        return s
    return s[-1] + reverse_str(s[:-1])  # last char + reverse of the rest
```

#### Factorial — Detailed Walkthrough

The recursive rule is:

- `factorial(0) = 1` (base case)
- `factorial(n) = n * factorial(n-1)` for `n > 0`

For `factorial(5)`, the call phase and return phase are different:

```text
Call phase:
factorial(5) -> factorial(4) -> factorial(3) -> factorial(2) -> factorial(1) -> factorial(0)

Return phase:
factorial(0) = 1
factorial(1) = 1 * 1 = 1
factorial(2) = 2 * 1 = 2
factorial(3) = 3 * 2 = 6
factorial(4) = 4 * 6 = 24
factorial(5) = 5 * 24 = 120
```

This is why recursion is conceptually two-step: **go down** to the base case, then **build up** the final answer while returning.

In [ ]:
def factorial_verbose(n, depth=0):
    """Print call/return trace, then return n!."""
    indent = "  " * depth
    print(f"{indent}call factorial({n})")

    if n == 0:
        print(f"{indent}return 1  (base case)")
        return 1

    sub = factorial_verbose(n - 1, depth + 1)
    result = n * sub
    print(f"{indent}return {n} * {sub} = {result}")
    return result

print("Tracing factorial(5):")
value = factorial_verbose(5)
print("Final result:", value)

In [ ]:
def countdown(n):
    if n <= 0:
        print("Go!")
        return
    print(n)
    countdown(n - 1)

def factorial(n):
    """Return n! recursively. n must be a non-negative integer."""
    if n == 0:
        return 1
    return n * factorial(n - 1)

def reverse_str(s):
    """Return the reverse of string s recursively."""
    if len(s) <= 1:
        return s
    return s[-1] + reverse_str(s[:-1])

countdown(5)
print()
print("5! =", factorial(5))    # 120
print("10! =", factorial(10))  # 3628800
print(reverse_str("hello"))    # 'olleh'

### 1.3 Stack Frames — How Python Executes Recursion

Python uses the **call stack** (the same LIFO structure from Week 6!) to execute recursive functions. Every time a function is called, Python pushes a new **frame** onto the stack. The frame stores:

- The local variables for that call
- The instruction pointer (where to return)

When the function returns, its frame is **popped** from the stack.

#### Call Stack Trace for `factorial(4)`

```
factorial(4)  ← pushed, waits for factorial(3)
  factorial(3)  ← pushed, waits for factorial(2)
    factorial(2)  ← pushed, waits for factorial(1)
      factorial(1)  ← pushed, waits for factorial(0)
        factorial(0)  ← BASE CASE → returns 1
      ← pops, returns 1 × 1 = 1
    ← pops, returns 2 × 1 = 2
  ← pops, returns 3 × 2 = 6
← pops, returns 4 × 6 = 24
```

Each indented level is one frame on the call stack. The maximum depth of the stack at any point equals the recursion depth.

#### Recursion Limit

Python enforces a default recursion limit of 1000 frames (`sys.getrecursionlimit()`). Exceeding it raises `RecursionError`. This is a feature, not a bug — it prevents accidental infinite recursion from crashing your machine.

In [ ]:
import sys

print("Default recursion limit:", sys.getrecursionlimit())

# Demonstrate the call stack depth
def depth_counter(n, current_depth=0):
    """Count recursion depth by passing it along."""
    if n == 0:
        return current_depth
    return depth_counter(n - 1, current_depth + 1)

print("Depth of factorial(10):", depth_counter(10), "frames")
print("Depth of factorial(100):", depth_counter(100), "frames")

### 1.4 Base Conversion

Converting an integer to a different base (e.g., binary or hexadecimal) produces remainders in *reverse* order. A stack naturally reverses a sequence — and recursion is equivalent to an implicit stack.

**Algorithm:**
- Base case: if `n < base`, return the single digit directly
- Recursive step: convert `n // base` first (higher-order digits), then append `n % base`

```
to_str(769, 10):
  769 // 10 = 76 → to_str(76, 10) → recurse
    76 // 10 = 7  → to_str(7, 10)  → base case → '7'
    76 %  10 = 6  → '6'
  769 % 10  = 9   → '9'
Result: '769'
```

In [ ]:
def to_str(n, base):
    """Convert non-negative integer n to a string in the given base (2-16)."""
    digits = '0123456789ABCDEF'
    if n < base:            # base case: single digit
        return digits[n]
    return to_str(n // base, base) + digits[n % base]

print(to_str(769, 10))   # '769'
print(to_str(10,  2))    # '1010'  (binary)
print(to_str(255, 16))   # 'FF'    (hex)
print(to_str(8,   2))    # '1000'

# Verify against built-in
assert to_str(255, 16) == 'FF'
assert to_str(10, 2) == bin(10)[2:].upper()
print("All assertions passed")

### 1.5 Visualising Recursion with Turtle (Optional)

Recursion becomes much easier to understand when you can **see** the repeated structure.

Python's `turtle` module is useful for this because:
- each recursive call draws a smaller version of the same shape,
- the base case stops drawing,
- the final picture makes the call pattern visible.

A common example is a **fractal tree**:
- draw one branch,
- recursively draw a smaller left branch,
- recursively draw a smaller right branch.

> **Exam note:** You are **not** expected to memorise turtle API details.  
> For assessment, focus on recursion logic: base case, smaller sub-problem, and self-calls.

#### If turtle does not work in your notebook

Some Jupyter kernels do not include `tkinter`, which is required by `turtle`.
If you get an error like `No module named '_tkinter'`, run the code as standalone script from a terminal instead:

```bash
python <your-pythonfil-ename>.py
```

In [ ]:
# Recursive turtle demo: fractal tree
# Note: this opens a GUI window in local Python environments.
import turtle


def tree(branch_len, t):
    if branch_len <= 5:      # base case
        return

    t.forward(branch_len)

    t.right(20)
    tree(branch_len - 15, t)  # recursive call 1

    t.left(40)
    tree(branch_len - 15, t)  # recursive call 2

    t.right(20)
    t.backward(branch_len)


screen = turtle.Screen()
screen.title("Recursive Fractal Tree")

pen = turtle.Turtle()
pen.left(90)
pen.speed("fastest")
pen.color("black")

tree(75, pen)
screen.exitonclick()

### 1.6 Tower of Hanoi (Book §4.5)

**Problem:** Move `n` disks from peg A to peg C, using peg B as auxiliary.

**Rules:**
- Only one disk may be moved at a time.
- A larger disk may never be placed on a smaller one.

**Why iteration is hard:** You need to track the state of three pegs simultaneously. The logic for n pegs involves an exponential number of cases.

**Recursive insight:** To move `n` disks from A to C:
1. Move the top `n−1` disks from A to B (recursion, using C as auxiliary)
2. Move disk `n` from A to C (base step)
3. Move `n−1` disks from B to C (recursion, using A as auxiliary)

This decomposes the problem into two identical sub-problems of size `n−1`.

**Move count:** 2ⁿ − 1 moves are required for n disks. This is provably optimal.

In [ ]:
def move_tower(n, from_peg, to_peg, aux_peg):
    """Print the moves to transfer n disks from from_peg to to_peg."""
    if n == 1:   # base case: move a single disk
        print(f"Move disk 1 from {from_peg} to {to_peg}")
        return
    move_tower(n - 1, from_peg, aux_peg, to_peg)    # step 1
    print(f"Move disk {n} from {from_peg} to {to_peg}")    # step 2
    move_tower(n - 1, aux_peg, to_peg, from_peg)    # step 3

print("Tower of Hanoi — n=3:")
move_tower(3, 'A', 'C', 'B')
print("\n(should be 2^3 - 1 = 7 moves)")

### 1.7 Fibonacci: Naive vs Memoised

The Fibonacci sequence is defined as:

```
fib(0) = 0
fib(1) = 1
fib(n) = fib(n-1) + fib(n-2)  for n ≥ 2
```

#### The Exponential Trap

Naive recursion recomputes the same sub-problems repeatedly:

```
fib(5)
├── fib(4)
│   ├── fib(3)         ← computed twice!
│   │   ├── fib(2)
│   │   └── fib(1)
│   └── fib(2)
└── fib(3)             ← computed twice!
    ├── fib(2)
    └── fib(1)
```

This leads to **O(2ⁿ)** total calls — `fib(40)` requires over a billion recursive calls.

#### Memoisation: Cache What You've Already Computed

Store results in a dictionary. Before computing, check if the result is already cached.
This reduces the complexity from **O(2ⁿ) → O(n)**.

> **Preview:** Memoisation is the core idea behind **Dynamic Programming**, which you will study in Week 11.

In [ ]:
# Naive: O(2^n) — definition only (counting calls is Exercise 2.4)
def fib_naive_demo(n: int) -> int:
    if n <= 1:
        return n
    return fib_naive_demo(n - 1) + fib_naive_demo(n - 2)


# Memoised: O(n)
def fib_memo(n, cache=None):
    if cache is None:
        cache = {}
    if n in cache:
        return cache[n]
    if n <= 1:
        return n
    cache[n] = fib_memo(n - 1, cache) + fib_memo(n - 2, cache)
    return cache[n]


print("Fibonacci values (both methods should agree):")
for n in [0, 1, 5, 10, 20]:
    assert fib_memo(n) == fib_naive_demo(n), f"Mismatch at n={n}"
    print(f"  fib({n}) = {fib_memo(n)}")
print("All values match!")

---

## Part 2: Exercises

Work through **2.1–2.4** in order. Use plain Python. In Exercise 2.1, do **not** call `math.factorial` inside your own implementations — you may use `math.factorial` only in the separate **test** cell to check results. Avoid `functools.reduce` unless a task says otherwise. Add your own `assert` checks where they help.


### Exercise 2.1 — Factorial: Recursive and Iterative

**Part A.** Implement `factorial_recursive(n)` and `factorial_iterative(n)` from scratch (do not call `math.factorial`).

**Part B.** For both implementations, measure the call depth for n = 10, 100, 500. At what value of n does the recursive version crash? (Hint: use `sys.getrecursionlimit()`.)

**Part C.** Which version do you prefer for large n? Why?

In [ ]:
# Exercise 2.1 — implementations (run the next cell to test when ready)

def factorial_recursive(n: int) -> int:
    """Return n! using recursion. Raises ValueError for negative n."""
    # Hint: base case n == 0 → 1; else n * factorial_recursive(n - 1)
    raise NotImplementedError("Replace this with your recursive factorial.")


def factorial_iterative(n: int) -> int:
    """Return n! using iteration. Raises ValueError for negative n."""
    # Hint: start with result = 1, loop k from 2 to n (or 1 to n), result *= k
    raise NotImplementedError("Replace this with your iterative factorial.")


In [ ]:
# Exercise 2.1 — tests (OK to use math.factorial here only, for checking)
import math

for k in [0, 1, 5, 10]:
    assert factorial_recursive(k) == math.factorial(k), f"Recursive wrong at {k}"
    assert factorial_iterative(k) == math.factorial(k), f"Iterative wrong at {k}"
print("Basic tests passed")

# Part B — experiment with recursion depth vs sys.getrecursionlimit() here


### Exercise 2.2 — Reverse a String Recursively

Implement `reverse_string(s: str) -> str` **recursively** (no `s[::-1]`, no `reversed()`).

- Base case: a string of length 0 or 1 is already reversed
- Recursive step: the reversed string is the last character plus the reverse of everything before it

In [ ]:
# Exercise 2.2 — implementation (run the next cell to test)

def reverse_string(s: str) -> str:
    """Return the reverse of s using recursion."""
    # Hint: base case len(s) <= 1; else s[-1] + reverse_string(s[:-1])
    raise NotImplementedError("Replace this with your recursive reverse.")

In [ ]:
# Exercise 2.2 — tests
assert reverse_string("") == ""
assert reverse_string("a") == "a"
assert reverse_string("hello") == "olleh"
assert reverse_string("racecar") == "racecar"
print("All tests passed")

### Exercise 2.3 — Tower of Hanoi

**Part A.** Implement `move_tower(n, from_peg, to_peg, aux_peg)` that **returns a list of move strings** (instead of printing them), e.g. `['Move disk 1 from A to C', ...]`.

**Part B.** Call your function for n = 4 and print the sequence of moves. How many moves does it take?

**Part C.** Verify that the number of moves equals 2ⁿ − 1 for n = 1, 2, 3, 4, 5 using an `assert`.

In [ ]:
# Exercise 2.3 — implementation
# Same three steps as the printed move_tower in §1.6, but collect strings in lists:
#   n==1 → return [f"Move disk 1 from {from_peg} to {to_peg}"]
#   else → moves_small + [f"Move disk {n} ..."] + moves_small (with pegs permuted)

def move_tower(n, from_peg, to_peg, aux_peg) -> list:
    """Return the list of moves to transfer n disks from from_peg to to_peg."""
    raise NotImplementedError("Replace this with your list-returning move_tower.")

In [ ]:
# Exercise 2.3 — Parts B & C (run after your move_tower is implemented)

moves_4 = move_tower(4, 'A', 'C', 'B')
print(f"n=4: {len(moves_4)} moves")
for m in moves_4:
    print(" ", m)

for n in [1, 2, 3, 4, 5]:
    moves = move_tower(n, 'A', 'C', 'B')
    assert len(moves) == 2**n - 1, f"Expected {2**n - 1} moves for n={n}, got {len(moves)}"
print("Move count formula 2^n - 1 verified for n=1..5")

### Exercise 2.4 — Fibonacci: Naive Then Memoised

**Part A.** Implement `fib_naive(n)` using simple recursion (no cache).

**Part B.** Implement `fib_fast(n)` using a memoisation dictionary.

**Part C.** Count the number of recursive calls made by each version for n = 5, 10, 15, 20, 25. Present the results in a table and comment on the difference.

**Part D.** *(Optional)* Plot the call counts for n = 5 to 35 using `matplotlib`.

In [ ]:
# Exercise 2.4 — implementations (run the next cell to test)

def fib_naive(n: int) -> int:
    """Return fib(n) using naive recursion. O(2^n) time."""
    # Hint: base n <= 1 return n; else fib_naive(n-1) + fib_naive(n-2)
    raise NotImplementedError("Replace this with naive Fibonacci.")


def fib_fast(n: int, cache: dict | None = None) -> int:
    """Return fib(n) using memoisation. O(n) time."""
    # Hint: if cache is None: cache = {}; if n in cache: return cache[n]; ...
    raise NotImplementedError("Replace this with memoised Fibonacci.")

In [ ]:
# Exercise 2.4 — correctness, then Part C (call counts)

expected = [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]
for i, val in enumerate(expected):
    assert fib_naive(i) == val, f"fib_naive({i}) wrong"
    assert fib_fast(i) == val, f"fib_fast({i}) wrong"
print("Correctness OK — now measure call counts (Part C):")

# Part C — count fib_naive body entries with a wrapper (example pattern):
#   calls = 0
#   def fib_naive_counted(n):
#       nonlocal calls
#       calls += 1
#       ... same logic as fib_naive ...
#   Then reset calls = 0 before each n and print calls for n in [5, 10, 15, 20, 25].
# For fib_fast, counting recursive entries is optional; comment on O(n) vs exponential instead.
